# Notebook 01.1 — Aquisição IBGE: malhas territoriais e dados socioeconômicos

**Projeto:** Acessibilidade Geográfica às UBS de Teresina — roteiro computacional AE2SFCA  
**Programa:** MAPEPROF — Mestrado Profissional em Planejamento Urbano e Regional / IFPI  
**Autor:** Felipe Ramos Dantas  
**Orientador:** Prof. Dr. Antonio Joaquim da Silva  
**Coorientador:** Prof. Dr. Reurysson Chagas de Sousa Morais  
**Repositório:** https://github.com/felipedantas-pi/ae2sfca-ubs  
**Última atualização:** 2026-06-08

## Objetivo

Baixar e pré-processar os dados geoespaciais e tabulares do **IBGE** necessários ao pipeline AE2SFCA. O processo inclui a delimitação da área de estudo (zona urbana + buffer de 5 km restrito ao limite municipal), a aquisição das geometrias censitárias e a extração e limpeza dos microdados do Censo 2022.

## Saídas

Gravadas em `dados/externos/ibge/`:

| Arquivo | Descrição |
|---|---|
| `teresina_municipio.parquet` | Limite municipal de Teresina |
| `teresina_bairros.parquet` | Malha de bairros oficiais |
| `teresina_zonaUrbana_utm.parquet` | Perímetro da zona urbana (bairros dissolvidos) |
| `teresina_zonaUrbana_buffer5kClip_utm.parquet` | Área de estudo: ZU + buffer 5 km $\cap$ limite municipal |
| `teresina_setoresCensitariosUrbanos.parquet` | Setores censitários restritos à situação urbana |
| `teresina_gradeEstatistica_utm.parquet` | Grade estatística intersecionada com o limite municipal |

Gravado em `dados/intermediarios/01_malha/`:

| Arquivo | Descrição |
|---|---|
| `teresina_setoresCensitarios_DadosCompletos.parquet` | GeoDataFrame mestre fundido com dados socioeconômicos (renda, cor/raça, demografia) |

## Pré-requisitos

- Conexão de internet estável (downloads do IBGE podem chegar a centenas de MB).
- Notebook executado uma única vez por release dos dados IBGE.
- Tempo médio: ~10–15 min.

---

In [1]:
# ── 1. IMPORTAÇÕES E CONFIGURAÇÃO GLOBAL ────────────────────────────────────
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
import zipfile
import io

# Caminhos e CRS centralizados (ver src/mapeprof/config.py)
from mapeprof.config import (
    EXT_IBGE,        # dados/externos/ibge   — onde gravar as malhas
    INT_MALHA,       # dados/intermediarios/01_malha  — onde gravar o master
    CRS_METRICO,     # EPSG:31983 — SIRGAS 2000 / UTM 23S
    CRS_GEOGRAFICO,  # EPSG:4674  — SIRGAS 2000 geográfico
    criar_diretorios,
)

criar_diretorios()
print("Dependências e diretórios prontos.")

Dependências e diretórios prontos.


In [2]:
# ── 2. DOWNLOAD E PRÉ-PROCESSAMENTO DAS MALHAS TERRITORIAIS ─────────────────

print("🌍 Baixando malhas territoriais do IBGE (Piauí): municipio, bairro e setores censitários...")
url_base           = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
url_pi_municipios  = "malhas_municipais/municipio_2025/UFs/PI/PI_Municipios_2025.zip"
url_pi_bairros     = "malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/bairros/shp/UF/PI_bairros_CD2022.zip"
url_pi_setores     = "malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/setores/shp/UF/PI_setores_CD2022.zip"

# Malhas territoriais
gdf_pi_mun            = gpd.read_file(f"zip+{url_base}{url_pi_municipios}")
gdf_pi_bairro         = gpd.read_file(f"zip+{url_base}{url_pi_bairros}")
gdf_teresina_scUrbano = gpd.read_file(f"zip+{url_base}{url_pi_setores}")

# Grade estatística nacional
print("📊 Baixando grade estatística do IBGE...")
url_grade = "https://geoftp.ibge.gov.br/recortes_para_fins_estatisticos/grade_estatistica/censo_2022/grade_estatistica/grade_id66.zip"
gdf_gradeEstatistica  = gpd.read_file(f"zip+{url_grade}")

# Cadastro de Endereços
print("📊 Baixando CNEFE de Teresina do IBGE...")
url_cnefe_pi = "https://ftp.ibge.gov.br/Cadastro_Nacional_de_Enderecos_para_Fins_Estatisticos/Censo_Demografico_2022/Arquivos_CNEFE/CSV/Municipio/22_PI/"
MUNICIPIO = "2211001_TERESINA.zip"
url_cnefe_completa = f"{url_cnefe_pi}{MUNICIPIO}"

pd_the_cnefe = pd.read_csv(url_cnefe_completa, sep=";", compression='zip', low_memory=False)
pd_the_cnefe.columns = pd_the_cnefe.columns.str.lower()
geometria = gpd.points_from_xy(pd_the_cnefe['longitude'], pd_the_cnefe['latitude'])
gdf_the_cnefe = gpd.GeoDataFrame(pd_the_cnefe, geometry=geometria, crs="EPSG:4674")

print("Download concluídos!")

🌍 Baixando malhas territoriais do IBGE (Piauí): municipio, bairro e setores censitários...
📊 Baixando grade estatística do IBGE...
📊 Baixando CNEFE de Teresina do IBGE...
Download concluídos!


In [5]:
print("✂️ Filtrando Teresina e estruturando a Zona Urbana...")
gdf_teresina_mun      = gdf_pi_mun.query("NM_MUN == 'Teresina'").copy()
gdf_teresina_bairros  = gdf_pi_bairro.query("NM_MUN == 'Teresina'").copy()
gdf_teresina_scUrbano = gdf_teresina_scUrbano.query("NM_MUN == 'Teresina' and SITUACAO == 'Urbana'")

print("🏠 Filtrando localização dos estabelecimentos do tipo domicilios...")
# No CNEFE 2022, a coluna oficial é cod_especie
gdf_the_cnefeDom     = gdf_the_cnefe.query("cod_especie in (1,2)").copy()

# Dissolve os limites internos dos bairros para gerar o polígono contínuo da Zona Urbana
print("🔲 Gerando o poligono da zona urbana de Teresina, a partir dos bairros...")
gdf_zonaUrbana = gdf_teresina_bairros.dissolve()[['CD_MUN', 'NM_MUN', 'geometry']]

✂️ Filtrando Teresina e estruturando a Zona Urbana...
🏠 Filtrando localização dos estabelecimentos do tipo domicilios...
🔲 Gerando o poligono da zona urbana de Teresina, a partir dos bairros...


In [7]:
# Construção da área de estudo: reprojeção métrica → buffer 5 km → reprojeção geográfica → clip
print("🔎 Criando um buffer de 5km ao redor da zona urbana para filtrar a malha viária no 'NB 01.2_Aquisicao_Malha_Overture.ipynb'")
gdf_zonaUrbana_5km = gdf_zonaUrbana.to_crs(CRS_METRICO)
gdf_zonaUrbana_5km['geometry'] = gdf_zonaUrbana_5km.buffer(distance=5000, cap_style='flat', join_style='bevel')
gdf_zonaUrbana_5km = gdf_zonaUrbana_5km.to_crs(CRS_GEOGRAFICO)

# Mantém a zona urbana expandida estritamente dentro do município de Teresina
gdf_zonaUrbana_5km_clip = gpd.clip(gdf_zonaUrbana_5km, gdf_teresina_mun)

# Filtra os estabelecimentos dentro da zona urbana
gdf_the_cnefe_dom_5km_clip = gpd.clip(gdf_the_cnefeDom, gdf_zonaUrbana)

🔎 Criando um buffer de 5km ao redor da zona urbana para filtrar a malha viária no 'NB 01.2_Aquisicao_Malha_Overture.ipynb'


In [8]:
# Exportação em GeoParquet, todos reprojetados para o CRS métrico
print("💾 Exportando arquivos territoriais...")
gdf_teresina_mun.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_municipio.parquet", index=False)
gdf_teresina_bairros.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_bairros.parquet", index=False)
gdf_teresina_scUrbano.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_setoresCensitariosUrbanos.parquet", index=False)
gdf_zonaUrbana.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_zonaUrbana_utm.parquet", index=False)
gdf_zonaUrbana_5km_clip.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_zonaUrbana_buffer5kClip_utm.parquet", index=False)
gdf_the_cnefe_dom_5km_clip.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_zonaUrbana_buffer5mClip_cnefeDom.parquet", index=False)

print("✅ Malhas territoriais processadas.")

💾 Exportando arquivos territoriais...
✅ Malhas territoriais processadas.


In [11]:
# ── 3. CRUZAMENTO ESPACIAL (SETORES E GRADE) ─────────────────────
print("✂️ Cruzando grade estatística com os limites de Teresina (overlay)...")
print("🔍 Selecionando células da grade estatística que tocam os limites de Teresina (sjoin)...")
# gdf_grade_intersec = gpd.overlay(
#     gdf_gradeEstatistica,
#     gdf_teresina_mun[['CD_MUN', 'NM_MUN', 'geometry']],
#     how='intersection'
# )
gdf_grade_intersec = gpd.sjoin(
    gdf_gradeEstatistica,
    gdf_teresina_mun[['CD_MUN', 'NM_MUN', 'geometry']],
    how='inner',
    predicate='intersects'
)
if 'index_right' in gdf_grade_intersec.columns:
    gdf_grade_intersec = gdf_grade_intersec.drop(columns=['index_right'])

print(f"✅ Seleção concluída. Total de células da grade preservadas intactas: {len(gdf_grade_intersec):,}")

✂️ Cruzando grade estatística com os limites de Teresina (overlay)...
🔍 Selecionando células da grade estatística que tocam os limites de Teresina (sjoin)...
✅ Seleção concluída. Total de células da grade preservadas intactas: 10,194


In [12]:
print("💾 Exportando arquivos censitários...")
gdf_grade_intersec.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_gradeEstatistica_utm.parquet", index=False)

print("✅ Geometrias censitárias processadas.")

💾 Exportando arquivos censitários...
✅ Geometrias censitárias processadas.


In [14]:
# ── 4. DECLARAÇÃO DE FUNÇÕES E URLS DO PIPELINE ETL ──────────────────────────

print("🌍 Baixando dados estatísticos do censo 2022 para as malhas territoriais do IBGE (Piauí)...")
URL_BASE_SETORES = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Setor_csv/"
URL_BASE_RENDA   = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios_Rendimento_do_Responsavel/"

# Dicionário de configuração das variáveis extraídas
TABELAS_CENSO = {
    "demografia": {
        "url": f"{URL_BASE_SETORES}Agregados_por_setores_demografia_BR.zip",
        "vars": {"V01006": "populacao"},                  # nº de moradores -> P_i
    },
    "renda": {
        "url": f"{URL_BASE_RENDA}Agregados_por_setores_renda_responsavel_BR_20260508_csv.zip",
        "vars": {"V06006": "renda_mediana",               # mediana -> indicador da H2
                 "V06004": "renda_media"},                # média -> comparação/robustez
}}

def extrair_e_limpar_censo(url, mapa_vars):
    """
    Baixa o CSV em memória, seleciona só as variáveis essenciais, padroniza a chave
    CD_SETOR (que vem ora 'CD_SETOR', ora 'CD_setor') e trata os marcadores do IBGE:
    '-' (zero real) -> 0 e 'X' (omitido por sigilo) -> NaN. NÃO usamos fillna(0):
    manter o NaN preserva a honestidade do sigilo e evita criar "renda zero" falsa
    em setores que na verdade têm dado protegido (ex.: condomínios de alto padrão).
    """
    print(f"  -> {', '.join(mapa_vars.values())}")
    resposta = requests.get(url)
    if resposta.status_code != 200:
        raise ValueError(f"❌ IBGE retornou HTTP {resposta.status_code}\nLink: {url}")

    with zipfile.ZipFile(io.BytesIO(resposta.content)) as z:
        nome_csv = next(n for n in z.namelist() if n.endswith(".csv"))
        with z.open(nome_csv) as f:
            # mapa upper->real resolve a inconsistência de caixa do IBGE (CD_setor vs CD_SETOR)
            cabecalho = pd.read_csv(f, sep=";", nrows=0).columns
            real = {c.upper(): c for c in cabecalho}
            col_chave = real["CD_SETOR"]
            cols = [col_chave] + [real[v] for v in mapa_vars]
            f.seek(0)
            df = pd.read_csv(f, sep=";", usecols=cols, dtype={col_chave: str})

    # padroniza a chave e renomeia para os nomes intuitivos
    df = df.rename(columns={col_chave: "CD_SETOR",
                            **{real[v]: nome for v, nome in mapa_vars.items()}})

    # limpeza dos marcadores do IBGE: '-' -> 0 ; 'X' -> NaN (sem fillna)
    for nome in mapa_vars.values():
        df[nome] = pd.to_numeric(df[nome].replace({"-": 0, "X": np.nan}), errors="coerce")
    return df

🌍 Baixando dados estatísticos do censo 2022 para as malhas territoriais do IBGE (Piauí)...


In [15]:
# ── 5. EXECUÇÃO DO PIPELINE ETL E MERGE ESPACIAL ────────────────────────────
print("📥 Extraindo e fundindo os dados tabulares (Censo 2022)...")
gdf_master = gdf_teresina_scUrbano.copy()

for chave, config in TABELAS_CENSO.items():
    df_temp = extrair_e_limpar_censo(config["url"], config["vars"])
    gdf_master = gdf_master.merge(df_temp, on="CD_SETOR", how="left")

# QA: setores sem dado utilizável. NÃO é erro de merge. São dois casos de sigilo:
#  (a) setores especiais sem linha em nenhuma tabela (CD_TIPO=4, condomínios etc.);
#  (b) setores com linha, mas com valores 'X' (omitidos), agora corretamente NaN.
# Listamos para rastreabilidade (Objetivo 1: reprodutibilidade).
sem_dado = gdf_master.loc[gdf_master["renda_mediana"].isna(), "CD_SETOR"].tolist()
print(f"\n⚠️  {len(sem_dado)} setores sem dado socioeconômico (sigilo IBGE), excluídos das análises:")
print(sem_dado)

caminho_final = INT_MALHA / "teresina_setoresCensitarios_DadosCompletos.parquet"
print(f"\n🎉 GeoDataFrame mestre: {len(gdf_master)} setores × {len(gdf_master.columns)} atributos.")
print(f"   válidos: {gdf_master['renda_mediana'].notna().sum()} | sem dado: {len(sem_dado)}")

📥 Extraindo e fundindo os dados tabulares (Censo 2022)...
  -> populacao


<positron-console-cell-15>:41: DtypeWarning: Columns (0: V01006) have mixed types. Specify dtype option on import or set low_memory=False.


  -> renda_mediana, renda_media

⚠️  35 setores sem dado socioeconômico (sigilo IBGE), excluídos das análises:
['221100105060169', '221100105060176', '221100105060193', '221100105060195', '221100105060201', '221100105060204', '221100105070002', '221100105070033', '221100105070292', '221100105070294', '221100105070296', '221100105070366', '221100105070368', '221100105070388', '221100105070390', '221100105080121', '221100105080226', '221100105080272', '221100105080306', '221100105080316', '221100105080326', '221100105080327', '221100105080330', '221100105080344', '221100105080349', '221100105080356', '221100105080362', '221100105090169', '221100105090239', '221100105100061', '221100105100224', '221100105100275', '221100105100282', '221100105100285', '221100105100353']

🎉 GeoDataFrame mestre: 1403 setores × 33 atributos.
   válidos: 1368 | sem dado: 35


In [16]:
print("💾 Exportando arquivos censitários...")
gdf_master.to_crs(CRS_METRICO).to_parquet(caminho_final, index=False)
print(f"Arquivo salvo em: {caminho_final}")

print("\n🎉 NB 01.1 Finalizado")

💾 Exportando arquivos censitários...
Arquivo salvo em: C:\Users\felipe\workspace_pcksa\ae2sfca_ubs\dados\intermediarios\01_malha\teresina_setoresCensitarios_DadosCompletos.parquet

🎉 NB 01.1 Finalizado
